# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset, "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells," using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema and is accessible via its official URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review all available record sets, their fields, columns, and associated `@id`s. This helps you identify which record set(s) to analyze.

In [ ]:
# List available record sets and their fields
print("Available record sets in this dataset:")
for record_set in metadata.record_set:
    print(f"- RecordSet @id: {record_set['@id']}, name: {record_set.get('name', '<no name>')}")
    fields = record_set.get('field', [])
    # Fields could be a dict or list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields and columns in this record set:")
    for field in fields:
        print(f"    Field @id: {field['@id']}, name: {field.get('name', '<no name>')}, column: {field.get('column', '<none>')}")
    print("")
if not getattr(metadata, 'record_set', []):
    print("No record set available in top-level metadata. Attempting to infer available file objects...")
    # Try to list available distributions (data files) if no record sets are defined
    for dist in getattr(metadata, 'distribution', []):
        print(f"- Distribution @id: {dist['@id']}, encoding: {dist.get('encodingFormat')}, url: {dist.get('contentUrl', dist.get('url', '<no url>'))}")

## 3. Data Extraction
Load data from a selected record set into a pandas DataFrame for further analysis. Use the record set and field `@id`s from the overview step.

In [ ]:
# ---
# Since the record sets are not present in `metadata.record_set`,
# we'll try to extract available record sets from the Dataset object
record_set_ids = [r['@id'] for r in getattr(metadata, 'record_set', [])]
if not record_set_ids:
    # Fallback: try known @id from the dataset's available file/distribution
    record_set_ids = [] # Fill if you know concrete record set @ids

# Print for reference
print("Record sets for extraction:", record_set_ids)

dataframes = {}
# The following logic is robust across datasets; if you know the right record_set_id, set it below
# If none are present, you could try using 'main' or a datafile @id

# If there are official record sets, extract them
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}: {df.columns.tolist()}")
    display(df.head())

# ---
# If the above did NOT find record sets, try using the default/first one
if not dataframes:
    # `mlcroissant` automatically selects the first record set if none is supplied
    print("No record_set @id explicitly available. Trying the default record set...")
    try:
        default_records = list(dataset.records())
        if default_records:
            df = pd.DataFrame(default_records)
            dataframes['default'] = df
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print("No records found in dataset.")
    except Exception as e:
        print(f"Error loading default records: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply simple transformations: filtering, normalization, and grouping. Use field `@id`s in DataFrame column access and calculations. **Set the `numeric_field_id` and `group_field_id` as appropriate for your dataset.**

In [ ]:
# Set the DataFrame to use
df_key = None
if dataframes:
    df_key = list(dataframes.keys())[0]
else:
    print("No DataFrame loaded; cannot proceed with EDA.")

if df_key is not None:
    df = dataframes[df_key]
    print(f"Available columns for EDA: {df.columns.tolist()}")
    # Set the @id of the numeric field and group field to analyze (replace with the actual values!!)
    numeric_field_id = None  # Example: 'cr:MW' (molecular weight) or similar, use the value from overview
    possible_numeric = [c for c in df.columns if str(c).startswith('cr:') or 'weight' in str(c).lower() or 'abundance' in str(c).lower()]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]
    else:
        numeric_field_id = df.columns[0] # fallback
    print(f"Using field (by @id/column) for numeric analysis: {numeric_field_id}")

    # Demo filtering: keep only rows where the numeric column > threshold
    threshold = 10
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the chosen numeric column
    mean = filtered_df[numeric_field_id].astype(float).mean()
    std = filtered_df[numeric_field_id].astype(float).std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a group field: often accession or some categorical name
    group_field_id = None
    possible_groups = [c for c in df.columns if 'desc' in str(c).lower() or 'accession' in str(c).lower() or 'cr:' in str(c)]
    if possible_groups:
        group_field_id = possible_groups[0]
    else:
        group_field_id = df.columns[0]

    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print(f"No appropriate group field found in columns; skipping grouping step.")

## 5. Visualization
Visualize distributions and relationships. Plots here use DataFrame column names referencing the fields via their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_key is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have demonstrated loading and exploring the FAIR² mass spectrometry dataset using `mlcroissant`.

- We loaded and parsed dataset metadata via the Croissant schema.
- Data records were explored using record set and field `@id`s.
- Several fields (columns) were filtered and normalized, and EDA/visualization was performed with clear mapping to dataset definitions.

For more advanced analysis, refer to the dataset's documentation, and adjust your analysis/plots to reference additional field or column `@id`s as needed.